In [1]:
RUN_TARGET = "colab"  # "colab" | "local"
# adjust OVERWRITE_EXISTING 

## Setup Instructions

Notebook 06 generates residue-level probe rows for supported active model families. It is registry-based rather than fully model-agnostic: unknown checkpoints are skipped, and retired top-1-unfrozen baseline checkpoints are intentionally excluded. Protein-level classification metrics are not generated here; notebook 07 loads them later from saved JSON artifacts.

### Typical workflow
1. Train or update a supported checkpoint in notebook 03, 04, or 05.
2. Run this notebook to generate or refresh saved probe-row CSVs in `results/probing/rows/`.
3. Open `07_compare_all_model_probes.ipynb` to build paper figures and tables from those saved artifacts.


In [2]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "pandas": "2.3.2",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "seaborn": "0.13.2",
        "captum": None,
    }

    def _version_matches(name: str, expected: str | None) -> bool:
        try:
            installed = _md.version(name)
            return installed == expected if expected is not None else True
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}" if version is not None else name
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Colab environment already compatible. No reinstall needed.


In [3]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


Mounted at /content/drive
Google Drive mounted: /content/drive/MyDrive/XAllergen2.0


# 06 - Registry-Driven Probe Row Generation

This notebook centralizes residue-level probe generation for the supported active model registry. It is semi-automatic over known model families, not a generic sweep over arbitrary checkpoint names. Unknown checkpoints are skipped, retired top-1-unfrozen baselines are ignored, and supported MTL probe rows are generated against the frozen baseline reference.


In [4]:
print("Bootstrap cell version: 06-probe-bootstrap-v3")
import sys
import urllib.request
from pathlib import Path

RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"


def add_src_to_path(project_root: Path, prepend: bool = True) -> bool:
    src_dir = project_root / "src"
    package_dir = src_dir / "xallergen"
    if package_dir.exists():
        if str(src_dir) in sys.path:
            sys.path.remove(str(src_dir))
        if prepend:
            sys.path.insert(0, str(src_dir))
        else:
            sys.path.append(str(src_dir))
        return True
    return False


def ensure_colab_xallergen_package(runtime_root: Path, refresh: bool = False) -> Path:
    src_dir = runtime_root / "src"
    package_dir = src_dir / "xallergen"
    package_dir.mkdir(parents=True, exist_ok=True)
    for module_name in [
        "__init__.py",
        "baseline_notebook_utils.py",
        "deep_plant_allergy_utils.py",
        "mtl_epitope_notebook_utils.py",
        "plotting_paper_figures.py",
    ]:
        target = package_dir / module_name
        if refresh or not target.exists():
            urllib.request.urlretrieve(f"{RAW}/src/xallergen/{module_name}", target)
            print(f"Downloaded: {module_name}")
    if str(src_dir) in sys.path:
        sys.path.remove(str(src_dir))
    sys.path.insert(0, str(src_dir))
    return src_dir


import importlib
import json

if RUN_TARGET == "colab":
    candidate_roots = []
    if "DRIVE_ROOT" in globals():
        candidate_roots.append(DRIVE_ROOT)
    candidate_roots.extend([
        Path("/content/drive/MyDrive/XAllergen2.0"),
        Path("/content/XAllergen2.0"),
    ])

    RUNTIME_ROOT = None
    for _root in candidate_roots:
        if add_src_to_path(_root):
            RUNTIME_ROOT = _root
            print(f"Using xallergen source from {_root / 'src'}")
            break

    if RUNTIME_ROOT is None:
        RUNTIME_ROOT = Path("/content/XAllergen2.0")
        _src_dir = ensure_colab_xallergen_package(RUNTIME_ROOT, refresh=True)
        print(f"Bootstrapped xallergen source into {_src_dir}")
    elif RUNTIME_ROOT == Path("/content/XAllergen2.0"):
        _src_dir = ensure_colab_xallergen_package(RUNTIME_ROOT, refresh=True)
        print(f"Refreshed xallergen source in {_src_dir}")

    if str(RUNTIME_ROOT) in sys.path:
        sys.path.remove(str(RUNTIME_ROOT))
    sys.path.insert(0, str(RUNTIME_ROOT))
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if add_src_to_path(_candidate):
            break

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.deep_plant_allergy_utils as deep_plant_allergy_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils
import xallergen.plotting_paper_figures as plotting_paper_figures

importlib.reload(baseline_notebook_utils)
importlib.reload(deep_plant_allergy_utils)
importlib.reload(mtl_epitope_notebook_utils)
importlib.reload(plotting_paper_figures)

from xallergen.baseline_notebook_utils import (
    DROPOUT,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    build_tokenizer,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    load_mtl_checkpoint,
    prepare_baseline_probe_frame,
    print_runtime_context,
    run_baseline_probe_suite,
)
from xallergen.deep_plant_allergy_utils import (
    build_embedding_model,
    build_tokenizer as build_deep_plant_tokenizer,
    load_deep_plant_allergy_checkpoint,
    run_deep_plant_probe_suite,
)
from xallergen.mtl_epitope_notebook_utils import (
    get_retired_checkpoint_names,
    get_supported_probe_model_registry,
    run_probe_suite,
    summarize_probe_outputs,
)
from xallergen.plotting_paper_figures import (
    build_output_paths_for_supported_mtl,
    save_registry_probe_summary,
)

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd
import torch


import contextlib
import numpy as np
from captum.attr import IntegratedGradients


def _patch_runtime_attribution_helpers() -> None:
    def _cudnn_backward_safe_context(device: str):
        if str(device).startswith("cuda"):
            return torch.backends.cudnn.flags(enabled=False)
        return contextlib.nullcontext()

    def _safe_compute_integrated_gradients(
        model,
        tokenizer,
        sequence: str,
        device: str,
        steps: int = baseline_notebook_utils.IG_STEPS,
        normalize: bool = False,
        internal_batch_size: int | None = 1,
    ):
        encodings = baseline_notebook_utils.tokenize_sequence(tokenizer, sequence, device)
        attention_mask = encodings["attention_mask"]
        input_embeds = model.backbone.get_input_embeddings()(encodings["input_ids"]).detach()
        baseline = torch.zeros_like(input_embeds)

        def ig_forward(inputs_embeds: torch.Tensor) -> torch.Tensor:
            return model.forward_from_inputs_embeds(inputs_embeds, attention_mask)["logits"]

        with _cudnn_backward_safe_context(device):
            attributions = IntegratedGradients(ig_forward).attribute(
                inputs=input_embeds,
                baselines=baseline,
                n_steps=steps,
                internal_batch_size=internal_batch_size,
            )
        importance = attributions.abs().sum(dim=-1).squeeze(0).detach().cpu().numpy()
        valid_length = int(attention_mask.sum().item())
        importance = importance[:valid_length]
        return baseline_notebook_utils.normalize_scores(importance) if normalize else importance

    def _safe_compute_gradient_x_input_scores(model, tokenizer, sequence: str, device: str):
        model.eval()
        encodings = baseline_notebook_utils.tokenize_sequence(tokenizer, sequence, device)
        attention_mask = encodings["attention_mask"]
        model.zero_grad(set_to_none=True)
        input_embeds = model.backbone.get_input_embeddings()(encodings["input_ids"]).detach()
        input_embeds.requires_grad_(True)
        with _cudnn_backward_safe_context(device):
            logits = model.forward_from_inputs_embeds(input_embeds, attention_mask)["logits"]
            gradients = torch.autograd.grad(
                outputs=logits.sum(),
                inputs=input_embeds,
                retain_graph=False,
                create_graph=False,
                only_inputs=True,
            )[0]
        scores = (input_embeds * gradients).sum(dim=-1).abs().squeeze(0)
        valid_length = int(attention_mask.sum().item())
        scores = scores[:valid_length].detach().cpu().numpy().astype(np.float32)
        model.zero_grad(set_to_none=True)
        return scores

    def _safe_compute_smoothgrad_ig_scores(
        model,
        tokenizer,
        sequence: str,
        device: str,
        steps: int,
        n_samples: int = 10,
        noise_std: float = 0.05,
        internal_batch_size: int = 10,
    ):
        model.eval()
        encodings = baseline_notebook_utils.tokenize_sequence(tokenizer, sequence, device)
        attention_mask = encodings["attention_mask"]
        base_embeds = model.backbone.get_input_embeddings()(encodings["input_ids"]).detach()
        baseline = torch.zeros_like(base_embeds)
        noise_scale = float(noise_std) * float(base_embeds.detach().std().item())
        total_scores = np.zeros(int(attention_mask.sum().item()), dtype=np.float64)

        def ig_forward(inputs_embeds: torch.Tensor) -> torch.Tensor:
            return model.forward_from_inputs_embeds(inputs_embeds, attention_mask)["logits"]

        ig = IntegratedGradients(ig_forward)
        for _ in range(n_samples):
            model.zero_grad(set_to_none=True)
            noisy_embeds = base_embeds + torch.randn_like(base_embeds) * noise_scale if noise_scale > 0 else base_embeds
            with _cudnn_backward_safe_context(device):
                attributions = ig.attribute(
                    inputs=noisy_embeds.detach(),
                    baselines=baseline,
                    n_steps=steps,
                    internal_batch_size=internal_batch_size,
                )
            scores = attributions.abs().sum(dim=-1).squeeze(0).detach().cpu().numpy()
            total_scores += scores[: total_scores.shape[0]]
        model.zero_grad(set_to_none=True)
        return (total_scores / n_samples).astype(np.float32)

    def _safe_deep_plant_compute_integrated_gradients(
        model,
        embedding_model,
        tokenizer,
        sequence: str,
        device: str,
        steps: int = deep_plant_allergy_utils.IG_STEPS,
        normalize: bool = False,
        internal_batch_size: int | None = 1,
    ):
        residue_embeddings = deep_plant_allergy_utils.compute_residue_embeddings(embedding_model, tokenizer, sequence, device)
        baseline = torch.zeros_like(residue_embeddings)
        lstm_module = getattr(model, "lstm", None)
        lstm_training_state = None if lstm_module is None else lstm_module.training

        def ig_forward(inputs_embeds: torch.Tensor) -> torch.Tensor:
            return model(inputs_embeds).squeeze(-1)

        try:
            if lstm_module is not None:
                lstm_module.train()
            with _cudnn_backward_safe_context(device):
                attributions = IntegratedGradients(ig_forward).attribute(
                    inputs=residue_embeddings,
                    baselines=baseline,
                    n_steps=steps,
                    internal_batch_size=internal_batch_size,
                )
        finally:
            if lstm_module is not None and lstm_training_state is not None:
                lstm_module.train(lstm_training_state)
        importance = attributions.abs().sum(dim=-1).squeeze(0).detach().cpu().numpy()
        return deep_plant_allergy_utils.normalize_scores(importance) if normalize else importance

    baseline_notebook_utils.compute_integrated_gradients = _safe_compute_integrated_gradients
    baseline_notebook_utils.compute_gradient_x_input_scores = _safe_compute_gradient_x_input_scores
    baseline_notebook_utils.compute_smoothgrad_ig_scores = _safe_compute_smoothgrad_ig_scores
    deep_plant_allergy_utils.compute_integrated_gradients = _safe_deep_plant_compute_integrated_gradients
    mtl_epitope_notebook_utils.compute_integrated_gradients = _safe_compute_integrated_gradients
    mtl_epitope_notebook_utils.compute_gradient_x_input_scores = _safe_compute_gradient_x_input_scores
    mtl_epitope_notebook_utils.compute_smoothgrad_ig_scores = _safe_compute_smoothgrad_ig_scores
    print("Patched runtime attribution helpers for CUDA-safe backward passes.")


_patch_runtime_attribution_helpers()


Bootstrap cell version: 06-probe-bootstrap-v3
Downloaded: __init__.py
Downloaded: baseline_notebook_utils.py
Downloaded: deep_plant_allergy_utils.py
Downloaded: mtl_epitope_notebook_utils.py
Downloaded: plotting_paper_figures.py
Bootstrapped xallergen source into /content/XAllergen2.0/src
Patched runtime attribution helpers for CUDA-safe backward passes.


In [5]:
if RUN_TARGET == "colab":
    DRIVE_PROJECT_ROOT = DRIVE_ROOT if "DRIVE_ROOT" in globals() else Path("/content/drive/MyDrive/XAllergen2.0")
    PROJECT_ROOT = DRIVE_PROJECT_ROOT if DRIVE_PROJECT_ROOT.exists() else RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if "DRIVE_MODELS" in globals() and DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if "DRIVE_RESULTS" in globals() and DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
    print(f"Device: {DEVICE}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for _output_subdir in [
    RESULTS_DIR / "classification",
    RESULTS_DIR / "probing" / "rows",
    RESULTS_DIR / "probing" / "summaries",
    RESULTS_DIR / "figures" / "diagnostics",
]:
    _output_subdir.mkdir(parents=True, exist_ok=True)

POSITIVES_CSV = DATA_DIR / "positives_splitB.csv"
tokenizer = build_tokenizer(HF_MODEL_NAME)
epitope_probe_df = prepare_baseline_probe_frame(POSITIVES_CSV)
print(f"Probe proteins in splitB: {len(epitope_probe_df)}")


Project root: /content/drive/MyDrive/XAllergen2.0
Using models dir: /content/drive/MyDrive/XAllergen2.0/models
Using results dir: /content/drive/MyDrive/XAllergen2.0/results
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Probe proteins in splitB: 61


## Generation Parameters

- `OVERWRITE_EXISTING = False` reuses finished probe tables and fills only missing methods.
- `N_RANDOM_DRAWS` controls the random-null estimate written into the probe rows.
- `IG_INTERNAL_BATCH_SIZE` controls the memory/speed tradeoff for Captum IG and SmoothGrad-IG.
- `INCLUDE_INTEGRATED_GRADIENTS`, `INCLUDE_GRADIENT_X_INPUT`, and `INCLUDE_SMOOTHGRAD_IG` let you include or exclude the expensive gradient-based methods.


In [6]:
OVERWRITE_EXISTING = False
N_RANDOM_DRAWS = 100
IG_INTERNAL_BATCH_SIZE = 16
SMOOTHGRAD_IG_SAMPLES = 10
SMOOTHGRAD_IG_NOISE_STD = 0.05
PROGRESS_PRINT_EVERY = 1

INCLUDE_INTEGRATED_GRADIENTS = True
INCLUDE_GRADIENT_X_INPUT = False
INCLUDE_SMOOTHGRAD_IG = False


## Supported Registry

The supported-model registry defines which active model families notebook 06 knows how to probe. Unknown checkpoints are skipped rather than inferred automatically. Optional `MTL ESM-2 top-1` is supplementary only, and retired top-1-unfrozen baseline checkpoints are intentionally excluded.


In [7]:
retired_checkpoint_names = set(get_retired_checkpoint_names())
supported_registry = get_supported_probe_model_registry(
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    include_supplementary=True,
)
supported_lookup = {spec["checkpoint_name"]: spec for spec in supported_registry}
registry_df = pd.DataFrame(
    [
        {
            "family_key": spec["family_key"],
            "display_label": spec["display_label"],
            "checkpoint_name": spec["checkpoint_name"],
            "checkpoint_exists": spec["checkpoint_exists"],
            "probe_rows_path": str(spec["probe_rows_path"]),
            "summary_path": str(spec["summary_path"]),
            "supplementary_only": spec["supplementary_only"],
            "supported_methods": ", ".join(spec["supported_methods"]),
        }
        for spec in supported_registry
    ]
)
registry_df


,family_key,display_label,checkpoint_name,checkpoint_exists,probe_rows_path,summary_path,supplementary_only,supported_methods
0,baseline,Frozen ESM-2,baseline_frozen_esm2.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,False,"attention_weights, integrated_gradients, gradi..."
1,mtl_frozen,MTL ESM-2,mtl_frozen_esm2_epitope.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,False,"residue_head, attention_weights, integrated_gr..."
2,deep_plant_benchmark,DeepPlantAllergy,deep_plant_allergy_benchmark.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,False,"attention_weights, integrated_gradients, rando..."
3,mtl_top1_unfrozen,MTL ESM-2 top-1,mtl_top1_unfrozen_esm2_epitope.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,True,"residue_head, attention_weights, integrated_gr..."


## Probe Generation

The frozen baseline probe rows are generated first and then reused by supported MTL families. If a probe CSV already exists and `OVERWRITE_EXISTING = False`, notebook 06 now detects missing methods automatically and recomputes only those methods instead of rerunning successful probe results.


In [8]:
print("Main generation cell version: 06-probe-generation-v3")
import inspect

generation_records = []
deep_plant_embedding_model = None
deep_plant_tokenizer = None
baseline_probe_df = None


def _available_methods_from_rows(probe_rows_path: Path) -> str:
    if not Path(probe_rows_path).exists():
        return ""
    probe_df = pd.read_csv(probe_rows_path, usecols=["method"])
    methods = sorted(str(value) for value in probe_df["method"].dropna().unique())
    return ", ".join(methods)


def _append_record(spec: dict, status: str, probe_rows_path: Path | None = None) -> None:
    generation_records.append(
        {
            "checkpoint_name": spec["checkpoint_name"],
            "family_key": spec["family_key"],
            "display_label": spec["display_label"],
            "status": status,
            "available_methods": _available_methods_from_rows(probe_rows_path or spec["probe_rows_path"]),
            "probe_rows_path": str(probe_rows_path or spec["probe_rows_path"]),
        }
    )


def _supports_progress_label(fn) -> bool:
    return "progress_label" in inspect.signature(fn).parameters


def _supports_enabled_methods(fn) -> bool:
    return "enabled_methods" in inspect.signature(fn).parameters


def _supports_progress_print_every(fn) -> bool:
    return "progress_print_every" in inspect.signature(fn).parameters


def _selected_methods_for_spec(spec: dict) -> list[str]:
    methods = []
    for method in spec["supported_methods"]:
        if method == "integrated_gradients" and not INCLUDE_INTEGRATED_GRADIENTS:
            continue
        if method == "gradient_x_input" and not INCLUDE_GRADIENT_X_INPUT:
            continue
        if method == "smoothgrad_ig" and not INCLUDE_SMOOTHGRAD_IG:
            continue
        methods.append(method)
    return methods


def _format_method_list(methods: set[str] | list[str]) -> str:
    methods = list(methods)
    return ", ".join(sorted(str(method) for method in methods)) if methods else "none"


def _load_existing_probe_df(path: Path) -> pd.DataFrame:
    if not Path(path).exists():
        return pd.DataFrame()
    return mtl_epitope_notebook_utils.ensure_label_variant_column(pd.read_csv(path))


def _missing_methods(existing_df: pd.DataFrame, expected_methods: list[str]) -> set[str]:
    if existing_df.empty:
        return set(expected_methods)
    existing_methods = set(existing_df["method"].astype(str))
    return set(expected_methods) - existing_methods


def _restrict_to_methods(frame: pd.DataFrame, methods: set[str] | list[str]) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    method_set = {str(method) for method in methods}
    return frame.loc[frame["method"].astype(str).isin(method_set)].copy()


def _merge_rows(existing_df: pd.DataFrame, new_df: pd.DataFrame, replaced_methods: set[str]) -> pd.DataFrame:
    new_df = _restrict_to_methods(new_df, replaced_methods)
    if existing_df.empty:
        merged = new_df.copy()
    else:
        preserved = existing_df.loc[~existing_df["method"].astype(str).isin(replaced_methods)].copy()
        merged = pd.concat([preserved, new_df], ignore_index=True)
    if merged.empty:
        return merged
    merged = mtl_epitope_notebook_utils.ensure_label_variant_column(merged)
    dedupe_cols = ["accession", "method", "model_family", "label_variant"]
    if all(col in merged.columns for col in dedupe_cols):
        merged = merged.drop_duplicates(subset=dedupe_cols, keep="last").reset_index(drop=True)
    mtl_epitope_notebook_utils.validate_unique_probe_rows(merged)
    return merged


BASELINE_SUPPORTS_PROGRESS = _supports_progress_label(run_baseline_probe_suite)
DEEP_PLANT_SUPPORTS_PROGRESS = _supports_progress_label(run_deep_plant_probe_suite)
MTL_SUPPORTS_PROGRESS = _supports_progress_label(run_probe_suite)
BASELINE_SUPPORTS_ENABLED_METHODS = _supports_enabled_methods(run_baseline_probe_suite)
DEEP_PLANT_SUPPORTS_ENABLED_METHODS = _supports_enabled_methods(run_deep_plant_probe_suite)
MTL_SUPPORTS_ENABLED_METHODS = _supports_enabled_methods(run_probe_suite)
BASELINE_SUPPORTS_PROGRESS_PRINT_EVERY = _supports_progress_print_every(run_baseline_probe_suite)
DEEP_PLANT_SUPPORTS_PROGRESS_PRINT_EVERY = _supports_progress_print_every(run_deep_plant_probe_suite)
MTL_SUPPORTS_PROGRESS_PRINT_EVERY = _supports_progress_print_every(run_probe_suite)

baseline_spec = next(spec for spec in supported_registry if spec["family_key"] == "baseline")
baseline_expected_methods = _selected_methods_for_spec(baseline_spec)
existing_baseline_probe_df = _load_existing_probe_df(baseline_spec["probe_rows_path"])
baseline_missing_methods = set() if OVERWRITE_EXISTING else _missing_methods(existing_baseline_probe_df, baseline_expected_methods)
if OVERWRITE_EXISTING:
    baseline_missing_methods = set(baseline_expected_methods)

if baseline_spec["checkpoint_exists"]:
    if not baseline_missing_methods and not OVERWRITE_EXISTING:
        baseline_probe_df = existing_baseline_probe_df.copy()
        baseline_probe_df["model_family"] = baseline_spec["display_label"]
        print(f"Reusing baseline probe rows: {baseline_spec['probe_rows_path']}")
        _append_record(baseline_spec, "reused")
    else:
        baseline_model, _ = load_baseline_checkpoint(baseline_spec["checkpoint_path"], DEVICE)
        print(
            f"Starting {baseline_spec['display_label']} on {len(epitope_probe_df)} proteins; missing methods: {_format_method_list(baseline_missing_methods if not OVERWRITE_EXISTING else baseline_expected_methods)}"
        )
        baseline_kwargs = dict(
            model=baseline_model,
            tokenizer=tokenizer,
            eval_df=epitope_probe_df,
            device=DEVICE,
            ig_steps=IG_STEPS,
            n_random_draws=N_RANDOM_DRAWS,
            ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
            smoothgrad_ig_samples=SMOOTHGRAD_IG_SAMPLES,
            smoothgrad_ig_noise_std=SMOOTHGRAD_IG_NOISE_STD,
            include_shuffled_mean=False,
        )
        if BASELINE_SUPPORTS_PROGRESS:
            baseline_kwargs["progress_label"] = f"Evaluating proteins [{baseline_spec['display_label']}]"
        else:
            print(f"Evaluating proteins [{baseline_spec['display_label']}]")
        if BASELINE_SUPPORTS_ENABLED_METHODS:
            baseline_kwargs["enabled_methods"] = baseline_missing_methods if not OVERWRITE_EXISTING else set(baseline_expected_methods)
        if BASELINE_SUPPORTS_PROGRESS_PRINT_EVERY:
            baseline_kwargs["progress_print_every"] = PROGRESS_PRINT_EVERY
        regenerated_baseline_df = run_baseline_probe_suite(**baseline_kwargs)
        regenerated_baseline_df = _restrict_to_methods(
            regenerated_baseline_df,
            baseline_missing_methods if not OVERWRITE_EXISTING else set(baseline_expected_methods),
        )
        print(f"Finished {baseline_spec['display_label']}: wrote {len(regenerated_baseline_df)} probe rows before merge")
        regenerated_baseline_df["model_family"] = baseline_spec["display_label"]
        baseline_probe_df = _merge_rows(existing_baseline_probe_df, regenerated_baseline_df, baseline_missing_methods or set(baseline_expected_methods))
        baseline_probe_df.to_csv(baseline_spec["probe_rows_path"], index=False)
        save_registry_probe_summary(
            baseline_probe_df,
            baseline_spec["summary_path"],
            baseline_expected_methods,
        )
        print(f"Saved baseline probe rows to: {baseline_spec['probe_rows_path']}")
        _append_record(baseline_spec, "generated" if OVERWRITE_EXISTING or existing_baseline_probe_df.empty else "completed_missing")
else:
    print(f"Skipping missing supported checkpoint: {baseline_spec['checkpoint_name']}")
    _append_record(baseline_spec, "skipped_missing")

for spec in supported_registry:
    if spec["family_key"] == "baseline":
        continue
    if not spec["checkpoint_exists"]:
        print(f"Skipping missing supported checkpoint: {spec['checkpoint_name']}")
        _append_record(spec, "skipped_missing")
        continue

    expected_methods = _selected_methods_for_spec(spec)
    existing_probe_df = _load_existing_probe_df(spec["probe_rows_path"])
    missing_methods = set(expected_methods) if OVERWRITE_EXISTING else _missing_methods(existing_probe_df, expected_methods)

    if not missing_methods and not OVERWRITE_EXISTING:
        print(f"Reusing probe rows for {spec['display_label']}: {spec['probe_rows_path']}")
        _append_record(spec, "reused")
        continue

    if spec["model_kind"] == "deep_plant":
        print(
            f"Starting {spec['display_label']} on {len(epitope_probe_df)} proteins; missing methods: {_format_method_list(missing_methods if not OVERWRITE_EXISTING else expected_methods)}"
        )
        deep_plant_model, deep_plant_metadata = load_deep_plant_allergy_checkpoint(spec["checkpoint_path"], DEVICE)
        deep_plant_hf_model_name = deep_plant_metadata.get("hf_model_name")
        if deep_plant_embedding_model is None:
            deep_plant_tokenizer = build_deep_plant_tokenizer(deep_plant_hf_model_name)
            deep_plant_embedding_model = build_embedding_model(deep_plant_hf_model_name, device=DEVICE)
        deep_plant_kwargs = dict(
            model=deep_plant_model,
            embedding_model=deep_plant_embedding_model,
            tokenizer=deep_plant_tokenizer,
            eval_df=epitope_probe_df,
            device=DEVICE,
            ig_steps=IG_STEPS,
            n_random_draws=N_RANDOM_DRAWS,
        )
        if DEEP_PLANT_SUPPORTS_PROGRESS:
            deep_plant_kwargs["progress_label"] = f"Evaluating proteins [{spec['display_label']}]"
        else:
            print(f"Evaluating proteins [{spec['display_label']}]")
        if DEEP_PLANT_SUPPORTS_ENABLED_METHODS:
            deep_plant_kwargs["enabled_methods"] = missing_methods if not OVERWRITE_EXISTING else set(expected_methods)
        if DEEP_PLANT_SUPPORTS_PROGRESS_PRINT_EVERY:
            deep_plant_kwargs["progress_print_every"] = PROGRESS_PRINT_EVERY
        regenerated_probe_df = run_deep_plant_probe_suite(**deep_plant_kwargs)
        regenerated_probe_df = _restrict_to_methods(
            regenerated_probe_df,
            missing_methods if not OVERWRITE_EXISTING else set(expected_methods),
        )
        print(f"Finished {spec['display_label']}: wrote {len(regenerated_probe_df)} probe rows before merge")
        regenerated_probe_df["model_family"] = spec["display_label"]
        probe_df = _merge_rows(existing_probe_df, regenerated_probe_df, missing_methods or set(expected_methods))
        probe_df.to_csv(spec["probe_rows_path"], index=False)
        save_registry_probe_summary(
            probe_df,
            spec["summary_path"],
            expected_methods,
        )
        _append_record(spec, "generated" if OVERWRITE_EXISTING or existing_probe_df.empty else "completed_missing")
        continue

    output_paths = build_output_paths_for_supported_mtl(
        family_key=spec["family_key"],
        display_label=spec["display_label"],
        models_dir=MODELS_DIR,
        results_dir=RESULTS_DIR,
        baseline_checkpoint_path=baseline_spec["checkpoint_path"],
        baseline_summary_path=baseline_spec["summary_path"],
    )
    model, _ = load_mtl_checkpoint(
        spec["checkpoint_path"],
        DEVICE,
        model_name=HF_MODEL_NAME,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    )
    print(
        f"Starting {spec['display_label']} on {len(epitope_probe_df)} proteins; missing methods: {_format_method_list(missing_methods if not OVERWRITE_EXISTING else expected_methods)}"
    )
    mtl_kwargs = dict(
        model=model,
        tokenizer=tokenizer,
        epitope_probe_df=epitope_probe_df,
        baseline_checkpoint_path=baseline_spec["checkpoint_path"],
        device=DEVICE,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        output_paths=output_paths,
        ig_steps=IG_STEPS,
        n_random_draws=N_RANDOM_DRAWS,
        ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
        resume=not OVERWRITE_EXISTING,
        precomputed_baseline_probe_df=baseline_probe_df,
    )
    if MTL_SUPPORTS_PROGRESS:
        mtl_kwargs["progress_label"] = f"Probing splitB [{spec['display_label']}]"
    else:
        print(f"Probing splitB [{spec['display_label']}]")
    if MTL_SUPPORTS_ENABLED_METHODS:
        mtl_kwargs["enabled_methods"] = missing_methods if not OVERWRITE_EXISTING else set(expected_methods)
    if MTL_SUPPORTS_PROGRESS_PRINT_EVERY:
        mtl_kwargs["progress_print_every"] = PROGRESS_PRINT_EVERY
    probe_outputs = run_probe_suite(**mtl_kwargs)
    print(f"Finished {spec['display_label']}: saved {len(probe_outputs['probe_df'])} probe rows")
    summarize_probe_outputs(
        probe_df=probe_outputs["probe_df"],
        baseline_probe_df=probe_outputs["baseline_probe_df"],
        output_paths=output_paths,
    )
    _append_record(spec, "generated" if OVERWRITE_EXISTING or existing_probe_df.empty else "completed_missing", probe_rows_path=output_paths.probe_rows_path)

processed_checkpoint_names = {spec["checkpoint_name"] for spec in supported_registry}
for checkpoint_path in sorted(MODELS_DIR.glob("*.pt")):
    if checkpoint_path.name in processed_checkpoint_names:
        continue
    if checkpoint_path.name in retired_checkpoint_names:
        print(f"Skipping retired baseline checkpoint: {checkpoint_path.name}")
        generation_records.append(
            {
                "checkpoint_name": checkpoint_path.name,
                "family_key": "retired",
                "display_label": "Retired baseline",
                "status": "skipped_retired",
                "available_methods": "",
                "probe_rows_path": "",
            }
        )
        continue
    print(f"Skipping unknown checkpoint: {checkpoint_path.name}")
    generation_records.append(
        {
            "checkpoint_name": checkpoint_path.name,
            "family_key": "unknown",
            "display_label": "Unsupported checkpoint",
            "status": "skipped_unknown",
            "available_methods": "",
            "probe_rows_path": "",
        }
    )

generation_df = pd.DataFrame(generation_records)
generation_df


Main generation cell version: 06-probe-generation-v3
Reusing baseline probe rows: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/baseline_probing_rows.csv
Reusing probe rows for MTL ESM-2: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/mtl_probing_rows.csv
Starting DeepPlantAllergy on 61 proteins; missing methods: integrated_gradients


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Evaluating proteins [DeepPlantAllergy]


Evaluating proteins:   0%|          | 0/61 [00:00<?, ?it/s]

Finished DeepPlantAllergy: wrote 61 probe rows before merge
Reusing probe rows for MTL ESM-2 top-1: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/mtl_top1_unfrozen_probing_rows.csv
Skipping retired baseline checkpoint: baseline_top1_unfrozen_esm2.pt


,checkpoint_name,family_key,display_label,status,available_methods,probe_rows_path
0,baseline_frozen_esm2.pt,baseline,Frozen ESM-2,reused,"attention_weights, gradient_x_input, integrate...",/content/drive/MyDrive/XAllergen2.0/results/pr...
1,mtl_frozen_esm2_epitope.pt,mtl_frozen,MTL ESM-2,reused,"attention_weights, gradient_x_input, integrate...",/content/drive/MyDrive/XAllergen2.0/results/pr...
2,deep_plant_allergy_benchmark.pt,deep_plant_benchmark,DeepPlantAllergy,completed_missing,"attention_weights, integrated_gradients, rando...",/content/drive/MyDrive/XAllergen2.0/results/pr...
3,mtl_top1_unfrozen_esm2_epitope.pt,mtl_top1_unfrozen,MTL ESM-2 top-1,reused,"attention_weights, gradient_x_input, integrate...",/content/drive/MyDrive/XAllergen2.0/results/pr...
4,baseline_top1_unfrozen_esm2.pt,retired,Retired baseline,skipped_retired,,


## Generation Summary

The final summary table reports the checkpoint name, registry family key, publication label, generation status, methods present in the saved row artifact, and the probe-row output path.


In [9]:
summary_columns = [
    "checkpoint_name",
    "family_key",
    "display_label",
    "status",
    "available_methods",
    "probe_rows_path",
]
print(generation_df[summary_columns].to_string(index=False))


                  checkpoint_name           family_key    display_label            status                                                                                              available_methods                                                                                        probe_rows_path
          baseline_frozen_esm2.pt             baseline     Frozen ESM-2            reused               attention_weights, gradient_x_input, integrated_gradients, occlusion, random_mean, smoothgrad_ig                     /content/drive/MyDrive/XAllergen2.0/results/probing/rows/baseline_probing_rows.csv
       mtl_frozen_esm2_epitope.pt           mtl_frozen        MTL ESM-2            reused attention_weights, gradient_x_input, integrated_gradients, occlusion, random_mean, residue_head, smoothgrad_ig                          /content/drive/MyDrive/XAllergen2.0/results/probing/rows/mtl_probing_rows.csv
  deep_plant_allergy_benchmark.pt deep_plant_benchmark DeepPlantAllergy completed_missin

In [10]:
# %% [markdown]
# ## Optional: Disconnect Colab Runtime After Successful Completion

# %%
AUTO_DISCONNECT_COLAB = True

if AUTO_DISCONNECT_COLAB and IN_COLAB:
    print("Notebook finished successfully. Disconnecting and deleting Colab runtime to stop GPU usage.")
    from google.colab import runtime  # type: ignore
    runtime.unassign()

NameError: name 'IN_COLAB' is not defined